# BM3D denoising of the whole Harvard-GF dataset + upload to Hugging Face

Streams the Harvard-GF 200^3 OCT volumes, denoises **every B-scan** with **BM3D**, builds
consolidated `Training|Validation|Test_volumes.npy` (+ labels) in a staging folder and
uploads them to a Hugging Face dataset repo. Resume-safe: progress is saved after every
volume, so a disconnected run continues where it stopped.

## Execution model & honest timing expectations
- BM3D is a **CPU/OpenMP** algorithm (no GPU kernel). Colab GPUs do **not** speed it up.
  Only the number of CPU cores matters for throughput.
- Reference measurement (1 core, 200x200 B-scan, uint8): `lc` ~1.8 s/slice, `np` ~2.4 s/slice.
- Total: 3300 volumes x 200 B-scans = 660,000 slices. Required core-hours (approx, lc): ~330.

| CPU cores (workers) | lc wall-clock | note |
|---|---|---|
| 2 (Colab free T4) | ~7 days | impractical |
| 8-12 (Colab A100 runtime) | ~27-41 h | split over several sessions (resume) |
| 24 | ~14 h | good |
| 48+ (vast.ai CPU-heavy) | ~7 h | best for full run |

Set `CFG["workers"]` = number of CPU cores, run `RUN_BENCH` first to get a live estimate.


In [ ]:
import os, sys, json, io, time, zipfile, math
os.environ.setdefault("OMP_NUM_THREADS", "1")
import subprocess
def _pip(pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + pkgs, check=True)
try:
    import bm3d  # noqa: F401
    import huggingface_hub
    import numpy as np
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
except Exception:
    _pip(["bm3d", "huggingface_hub", "numpy", "matplotlib"])
    import numpy as np
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

try:
    from google.colab import userdata
    IN_COLAB = True
except Exception:
    userdata = None
    IN_COLAB = False

def hf_token():
    if os.environ.get("HF_TOKEN"):
        return os.environ["HF_TOKEN"]
    if userdata is not None:
        try:
            return userdata.get("HF_TOKEN")
        except Exception:
            return None
    return None


In [ ]:
CFG = {
    "hf_repo": "harvardairobotics/Harvard-GF",
    "zip_file": "Dataset/dataset.zip",
    "csv_file": "ReadMe/data_summary.csv",
    "staging": "/content/glaucoma_hf_200_bm3d",
    "out_repo": "",
    "out_private": True,
    "split_alias": {"training": "Training", "validation": "Validation",
                     "valid": "Validation", "test": "Test", "testing": "Test"},
    "profile": "lc",
    "sigma": 0.1,
    "workers": max(2, os.cpu_count() or 2),
    "limit_volumes": 0,
    "max_slices": 0,
    "save_every": 25,
    "run_bench": True,
    "abort_if_hours_gt": 120.0,
}
SPLITS = ("Training", "Validation", "Test")
RES = 200


In [ ]:
NAMES, META, ZIP_PATH = [], {}, None
def download_hf(filename):
    from huggingface_hub import hf_hub_download
    print(f"[data] downloading {CFG['hf_repo']}/{filename} ...", flush=True)
    return hf_hub_download(repo_id=CFG["hf_repo"], filename=filename, repo_type="dataset")
def ensure_meta():
    global NAMES, META, ZIP_PATH
    if META and NAMES:
        return
    import csv
    csv_path = download_hf(CFG["csv_file"])
    ZIP_PATH = download_hf(CFG["zip_file"])
    with open(csv_path, newline="") as fh:
        for r in csv.DictReader(fh):
            use = (r["use"] or "").strip().lower()
            split = CFG["split_alias"].get(use)
            if split is None:
                continue
            gl = 1 if str(r["glaucoma"]).strip().lower() in ("yes", "1", "true") else 0
            META[os.path.splitext(os.path.basename(r["filename"]))[0]] = (split, gl)
    with zipfile.ZipFile(ZIP_PATH) as zf:
        NAMES = [n for n in zf.namelist() if n.endswith(".npz")]
    NAMES = [n for n in NAMES if os.path.splitext(os.path.basename(n))[0] in META]
    NAMES.sort()
    print(f"[data] {len(META)} labeled | {len(NAMES)} zip entries kept")
def split_counts():
    c = {s: 0 for s in SPLITS}
    for n in NAMES:
        stem = os.path.splitext(os.path.basename(n))[0]
        c[META[stem][0]] += 1
    return c
def read_volume(entry):
    if ZIP_PATH is None:
        raise RuntimeError("ensure_meta() first")
    with zipfile.ZipFile(ZIP_PATH) as zf:
        return np.load(io.BytesIO(zf.read(entry)))["oct_bscans"]


In [ ]:
_PROFILE_OBJ = None
def set_denoiser(profile, sigma):
    global _PROFILE_OBJ, _SIGMA
    import bm3d
    _SIGMA = float(sigma)
    _PROFILE_OBJ = {"lc": bm3d.BM3DProfileLC(), "np": bm3d.BM3DProfile()}[profile]
def _bm3d_one(sl):
    import os
    os.environ.setdefault("OMP_NUM_THREADS", "1")
    import bm3d
    vmin, vmax = sl.min(), sl.max()
    if vmax - vmin < 1e-6:
        return sl.copy()
    z = (sl.astype(np.float32) - vmin) / (vmax - vmin)
    out = bm3d.bm3d(z, sigma_psd=_SIGMA, profile=_PROFILE_OBJ)
    return np.clip(np.asarray(out) * (vmax - vmin) + vmin, 0, 255).astype(np.uint8)
def make_pool(workers):
    if workers <= 1:
        return None
    import multiprocessing as mp
    try:
        ctx = mp.get_context("fork")
    except ValueError:
        return None
    from concurrent.futures import ProcessPoolExecutor
    return ProcessPoolExecutor(max_workers=workers, mp_context=ctx)
def denoise_slices(slices, workers):
    pool = make_pool(workers)
    if pool is None:
        return [_bm3d_one(s) for s in slices]
    try:
        return list(pool.map(_bm3d_one, slices))
    finally:
        pool.shutdown()
def bench_slice(profile):
    set_denoiser(profile, CFG["sigma"])
    rng = np.random.default_rng(0)
    z = rng.integers(0, 255, size=(RES, RES), dtype=np.uint8)
    t0 = time.time()
    _bm3d_one(z)
    return time.time() - t0
def estimate_hours(sec_per_slice, n_volumes, workers):
    return sec_per_slice * n_volumes * RES / workers / 3600.0


In [ ]:
def build_all(cfg):
    set_denoiser(cfg["profile"], cfg["sigma"])
    os.makedirs(cfg["staging"], exist_ok=True)
    counts = split_counts()
    print("[build] counts:", counts, flush=True)
    exists = os.path.exists(os.path.join(cfg["staging"], "Training_volumes.npy"))
    mode = "r+" if exists else "w+"
    vols, labs = {}, {}
    for s in SPLITS:
        n = counts[s]
        if n == 0:
            continue
        vols[s] = np.lib.format.open_memmap(os.path.join(cfg["staging"], f"{s}_volumes.npy"),
                                           mode=mode, dtype=np.uint8, shape=(n, 1, RES, RES, RES))
        labs[s] = np.lib.format.open_memmap(os.path.join(cfg["staging"], f"{s}_labels.npy"),
                                           mode=mode, dtype=np.int64, shape=(n,))
    prog_path = os.path.join(cfg["staging"], "progress.json")
    done = set(json.load(open(prog_path)).get("stems", [])) if os.path.exists(prog_path) else set()
    filled = {s: sum(1 for st in done if META[st][0] == s) for s in SPLITS}
    todo = [n for n in NAMES if os.path.splitext(os.path.basename(n))[0] not in done]
    if cfg["limit_volumes"] > 0:
        todo = todo[:cfg["limit_volumes"]]
    print(f"[build] resume: {len(done)}/{len(NAMES)} done; {len(todo)} to do", flush=True)
    t0, w0 = time.time(), len(done)
    for i, entry in enumerate(todo):
        stem = os.path.splitext(os.path.basename(entry))[0]
        split, label = META[stem]
        raw = read_volume(entry)
        slices = [raw[d] for d in range(raw.shape[0])]
        if cfg["max_slices"] > 0:
            slices = slices[:cfg["max_slices"]]
        den = denoise_slices(slices, cfg["workers"])
        if cfg["max_slices"] > 0 and len(den) < raw.shape[0]:
            den = den + [np.zeros_like(raw[0])] * (raw.shape[0] - len(den))
        vols[split][filled[split]] = np.stack(den)[None]
        labs[split][filled[split]] = label
        filled[split] += 1
        done.add(stem)
        if (len(done) - w0) % cfg["save_every"] == 0 or i == len(todo) - 1:
            for s in SPLITS:
                if s in vols:
                    vols[s].flush(); labs[s].flush()
            with open(prog_path, "w") as fh:
                json.dump({"stems": sorted(done)}, fh)
            el = time.time() - t0
            rate = el / max(len(done) - w0, 1)
            rem = rate * max(len(todo) - i - 1, 0)
            print(f"[build] {len(done)}/{len(NAMES)} | {rate:.2f} s/vol | ETA {rem/3600:.1f} h", flush=True)
    for s in SPLITS:
        if s in vols:
            vols[s].flush(); labs[s].flush()
    with open(prog_path, "w") as fh:
        json.dump({"stems": sorted(done)}, fh)
    with open(os.path.join(cfg["staging"], "manifest.json"), "w") as fh:
        json.dump({"source": cfg["hf_repo"], "denoiser": "BM3D",
                   "profile": cfg["profile"], "sigma": cfg["sigma"],
                   "resolution": RES, "per_bscan_minmax": True}, fh, indent=2)
    print("[build] DONE", flush=True)


In [ ]:
if CFG.get("run_bench", True):
    ensure_meta()
    nv = len(NAMES)
    for prof in ("lc", "np"):
        sec = bench_slice(prof)
        h = estimate_hours(sec, nv, CFG["workers"])
        print(f"[bench] profile={prof} {sec:.2f} s/slice -> ~{h:.1f} h at {CFG['workers']} workers", flush=True)
    sec = bench_slice(CFG["profile"])
    h = estimate_hours(sec, nv, CFG["workers"])
    print(f"[bench] chosen {CFG['profile']}: ~{h:.1f} h wall-clock", flush=True)
    if CFG["limit_volumes"] == 0 and h > CFG["abort_if_hours_gt"]:
        raise SystemExit(f"Projected {h:.1f}h > {CFG['abort_if_hours_gt']}h abort. "
                         f"Reduce budget, raise workers, or set run_bench=False.")


In [ ]:
GO_BUILD = bool(os.environ.get("GO_BUILD", "1") == "1")
if GO_BUILD:
    if CFG["limit_volumes"] > 0:
        print("[warn] trial run (limit_volumes>0): delete the staging folder before the full run", flush=True)
    ensure_meta()
    set_denoiser(CFG["profile"], CFG["sigma"])
    build_all(CFG)


In [ ]:
GO_UPLOAD = bool(os.environ.get("GO_UPLOAD", "1") == "1")
def whoami():
    from huggingface_hub import HfApi
    return HfApi(token=hf_token()).whoami()["name"]
if GO_UPLOAD:
    if not hf_token():
        raise SystemExit("HF_TOKEN missing (set env or Colab secret)")
    owner = whoami()
    repo = CFG["out_repo"] or f"{owner}/harvard-oct-glaucoma-200-bm3d"
    from huggingface_hub import HfApi
    api = HfApi(token=hf_token())
    api.create_repo(repo_id=repo, repo_type="dataset", private=CFG["out_private"], exist_ok=True)
    api.upload_folder(repo_id=repo, repo_type="dataset",
                     folder_path=CFG["staging"],
                     commit_message="Harvard-GF 200^3 BM3D-denoised")
    print("uploaded:", "https://huggingface.co/datasets/" + repo)


In [ ]:
def verify_outputs():
    ensure_meta()
    for s in SPLITS:
        p = os.path.join(CFG["staging"], f"{s}_volumes.npy")
        if os.path.exists(p):
            arr = np.load(p, mmap_mode="r")
            lbl = np.load(os.path.join(CFG["staging"], f"{s}_labels.npy"))
            print(s, "n=", arr.shape[0], "range", int(arr.min()), int(arr.max()),
                  "pos_rate", round(float(lbl.mean()), 3), flush=True)
    test_entry = next(n for n in NAMES if META[os.path.splitext(os.path.basename(n))[0]][0] == "Test")
    raw = read_volume(test_entry)
    den = np.asarray(np.load(os.path.join(CFG["staging"], "Test_volumes.npy"), mmap_mode="r")[0, 0])
    mse = float(np.mean((raw.astype(np.float32) - den.astype(np.float32)) ** 2))
    psnr = 10 * math.log10(255.0 ** 2 / max(mse, 1e-12))
    print(f"[verify] sample={os.path.basename(test_entry)} PSNR(raw,BM3D)={psnr:.2f} dB", flush=True)
    fig, axes = plt.subplots(2, 3, figsize=(12, 8))
    for j, mid in enumerate((60, 100, 140)):
        for r, (arr, t) in enumerate(((raw, "raw"), (den, "BM3D"))):
            im = arr[mid]
            p1, p99 = np.percentile(im, 1), np.percentile(im, 99)
            axes[r, j].imshow(im, cmap="gray", vmin=p1, vmax=p99)
            axes[r, j].set_title(f"{t} B-scan {mid}" if r == 0 else f"{t} B-scan {mid}")
            axes[r, j].axis("off")
    fig.suptitle(f"{os.path.basename(test_entry)} | BM3D {CFG['profile']} sigma={CFG['sigma']} | PSNR {psnr:.1f} dB")
    fig.tight_layout(rect=(0, 0, 1, 0.96))
    fig.savefig(os.path.join(CFG["staging"], "verify_bm3d_sample.png"), dpi=150)
    plt.close(fig)
    print("staging files:", sorted(os.listdir(CFG["staging"])), flush=True)
if os.path.exists(os.path.join(CFG["staging"], "Test_volumes.npy")):
    verify_outputs()
else:
    print("no staging output yet - run build first")
